# pulse-ep — interactive Jupyter walkthrough

End-to-end demo of consuming the **pulse-ep** REST API from a notebook.

1. Configure the connection from environment variables
2. Log in and list studies / maps
3. Fetch a mesh and visualise it interactively with PyVista
4. Plot the scalar distribution with matplotlib
5. Tabulate the area per score interval via `/calculate_areas_for_intervals`

Make sure the server is running (`pulse-ep-server` or `docker compose --profile server up`)
and that `PULSE_EP_BASE_URL`, `PULSE_EP_USERNAME`, `PULSE_EP_PASSWORD` are set.

## 1. Connection setup

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyvista as pv
import requests

BASE_URL = os.environ.get('PULSE_EP_BASE_URL', 'http://127.0.0.1:5000')
USERNAME = os.environ.get('PULSE_EP_USERNAME', 'admin')
PASSWORD = os.environ.get('PULSE_EP_PASSWORD')
if not PASSWORD:
    raise RuntimeError('PULSE_EP_PASSWORD env var is not set.')

# 'trame' gives an interactive viewer in JupyterLab but is an extra
# install; without this fallback the notebook died on a fresh install of
# pulse-ep, which does not pull it in. 'static' renders images instead.
try:
    pv.set_jupyter_backend('trame')
except Exception:
    pv.set_jupyter_backend('static')
    pv.OFF_SCREEN = True
BASE_URL

## 2. Authenticate

In [ ]:
def pe_login(base_url: str, username: str, password: str) -> str:
    r = requests.post(
        f'{base_url}/login_user',
        json={'username': username, 'password': password},
        timeout=15,
    )
    r.raise_for_status()
    return r.json()['access_token']


def pe_headers(token: str) -> dict[str, str]:
    return {'Authorization': f'Bearer {token}'}


token = pe_login(BASE_URL, USERNAME, PASSWORD)
print('Authenticated successfully.')

## 3. Browse studies and maps

In [ ]:
studies = pd.DataFrame(
    requests.get(f'{BASE_URL}/list_studies', headers=pe_headers(token), timeout=15).json()
)
studies.head(10)

In [ ]:
study_id = int(os.environ.get('PE_STUDY_ID', studies['id'].iloc[0]))
maps = pd.DataFrame(
    requests.get(
        f'{BASE_URL}/list_epmaps_in_study/{study_id}',
        headers=pe_headers(token), timeout=15,
    ).json()
)
maps.head(10)

## 4. Fetch one map's mesh

In [ ]:
map_id = int(os.environ.get('PE_MAP_ID', maps['id'].iloc[0]))
params = {'map_id': map_id, 'distance': 5.0}
payload = requests.get(
    f'{BASE_URL}/get_mesh_data', params=params, headers=pe_headers(token), timeout=60,
).json()

vertices = np.asarray(payload['mesh_data']['vertices'], dtype=float)
faces_raw = np.asarray(payload['mesh_data']['faces'], dtype=np.int64)
# The response names the quantity it used, so an omitted
# scalar_name still yields a correctly labelled plot.
scalar_name = payload['scalar_name']

scalars = np.asarray(
    [np.nan if v is None else float(v) for v in payload['mesh_data']['scalar_data']],
    dtype=float,
)

# PyVista's PolyData expects a flat connectivity array prefixed with '3' per triangle
faces = np.column_stack([np.full(len(faces_raw), 3, dtype=np.int64), faces_raw]).ravel()
mesh = pv.PolyData(vertices, faces)
mesh[scalar_name] = scalars
f'Loaded mesh: {mesh.n_points} vertices, {mesh.n_cells} faces, scalar non-NaN = {np.isfinite(scalars).sum()}'

## 5. Interactive 3D rendering

In [ ]:
pl = pv.Plotter(notebook=True)
pl.add_mesh(mesh, scalars=scalar_name, nan_color='lightgrey', cmap='turbo', smooth_shading=True)
pl.add_text(f'pulse-ep map {map_id}', position='upper_left')
pl.show()

## 6. Scalar distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(scalars[np.isfinite(scalars)], bins=40, color='#3a76ff', edgecolor='white')
ax.set_xlabel('Scalar value (act)')
ax.set_ylabel('Vertex count')
ax.set_title(f'pulse-ep map {map_id} — scalar distribution')
fig.tight_layout()
fig.show()

## 7. Area per interval

`/calculate_areas_for_intervals` returns the surface area that falls into
each bin — the same per-map reduction the bundled web viewer and the
clinical Excel reports compute. Re-computing it here from the raw REST
payload makes this notebook a Python contribution to the cross-language
reproducibility statement of the software paper.

**The bins must match the quantity.** A pace-mapping score runs 0–100 %, a
bipolar voltage is in mV and an activation map in ms, so a fixed 50–100
grid reports zero area everywhere on most maps. The bins below are derived
from the data actually returned.


In [ ]:
# Bins derived from the mesh values, so they fit whatever quantity this map
# carries. Replace with your own clinical grid, e.g. [[0, 0.5], [0.5, 1.5]]
# for a bipolar voltage map.
finite = scalars[np.isfinite(scalars)]
edges = np.linspace(finite.min(), finite.max(), 6)
intervals = [[float(lo), float(hi)] for lo, hi in zip(edges[:-1], edges[1:])]

payload = {
    'map_id': map_id,
    'distance': 5.0,
    'intervals': intervals,
}
areas_resp = requests.post(
    f'{BASE_URL}/calculate_areas_for_intervals',
    json=payload, headers=pe_headers(token), timeout=60,
).json()

areas_df = pd.DataFrame({
    'interval': [f'{lo:.3g}–{hi:.3g}' for lo, hi in intervals],
    'area_cm2': [None if a is None else float(a) for a in areas_resp['areas']],
})
print(f'{scalar_name}: total covered area = {areas_df["area_cm2"].sum():.3f} cm²')
areas_df
